# 03 Cascade SLU on real Whisper transcripts: SFT then GRPO (verifiable reward)

In [ ]:
# Settings: GPU T4 x2, Internet ON. Persistent outputs: results/ and runs/ in /kaggle/working.
import os, shutil, subprocess
REPO_URL = "https://github.com/ByteBoss123/speaker-aware-speech-llm.git"
WORK = "/kaggle/working/speaker-aware-speech-llm"
if not os.path.exists(WORK):
    src = "/kaggle/input/speaker-aware-speech-llm"   # alternative: upload the repo as a Kaggle dataset
    if os.path.exists(src):
        shutil.copytree(src, WORK)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
os.chdir(WORK)
DATA = "/tmp/data"   # large ephemeral disk; /kaggle/working is capped at ~20 GB
os.environ["DATA"] = DATA

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2

In [ ]:
!bash scripts/prepare_data.sh $DATA slurp

Whisper-small transcripts: 24k train recordings, 2k devel, full test (2-GPU shards)

In [ ]:
%%bash
for S in slurp_train slurp_devel slurp_test; do
  M=0; [ $S = slurp_train ] && M=24000; [ $S = slurp_devel ] && M=2000
  CUDA_VISIBLE_DEVICES=0 python scripts/asr_whisper.py --data_root $DATA --set $S --max_items $M --shard 0/2 --out results/asr > /tmp/$S.0.log 2>&1 &
  CUDA_VISIBLE_DEVICES=1 python scripts/asr_whisper.py --data_root $DATA --set $S --max_items $M --shard 1/2 --out results/asr > /tmp/$S.1.log 2>&1
  wait
  python scripts/asr_whisper.py --data_root $DATA --set $S --merge --out results/asr | tail -4
done

In [ ]:
!python scripts/slu_text.py sft --data_root $DATA --asr_dir results/asr --run_dir runs/slu

In [ ]:
!python scripts/slu_text.py grpo --data_root $DATA --asr_dir results/asr --run_dir runs/slu --grpo_steps 300

In [ ]:
!python scripts/slu_text.py eval --data_root $DATA --asr_dir results/asr --run_dir runs/slu --out results/slu

In [ ]:
!python scripts/summarize.py results runs
!cd /kaggle/working && zip -qr results_slu.zip speaker-aware-speech-llm/results speaker-aware-speech-llm/runs -x '*.npz'